# OpenMeteo Weather Data Analysis - Extended Dataset (2023-2025)

Analysis of weather data from OpenMeteo for the period June 2023 to June 2025.

**Version**: V3

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
data_path = '/Users/vojtech/Code/Bard89/Project-Data/data/processed/jp_openmeteo_processed_20230601_to_20250630.csv'
print(f"Loading OpenMeteo data from: {data_path}")

df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Loading OpenMeteo data from: /Users/vojtech/Code/Bard89/Project-Data/data/processed/jp_openmeteo_processed_20230601_to_20250630.csv


## 1. Dataset Overview

In [ ]:
print("Dataset Info:")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Unique hexagons (res8): {df['h3_index_res8'].nunique():,}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype}")

In [ ]:
print("First 10 rows:")
display(df.head(10))

print("\nLast 10 rows:")
display(df.tail(10))

## 2. Temporal Coverage Analysis

In [ ]:
df['date'] = df['timestamp'].dt.date
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month

all_dates = pd.date_range('2023-06-01', '2025-06-30', freq='D').date
existing_dates = set(df['date'].unique())
missing_dates = sorted(set(all_dates) - existing_dates)

print(f"Temporal Coverage Analysis:")
print("="*60)
print(f"Expected days: {len(all_dates)}")
print(f"Days with data: {len(existing_dates)}")
print(f"Missing days: {len(missing_dates)}")
print(f"Coverage: {len(existing_dates)/len(all_dates)*100:.1f}%")

if len(existing_dates) == len(all_dates):
    print("\n✓ COMPLETE COVERAGE CONFIRMED!")
else:
    print(f"\n⚠ Missing {len(missing_dates)} days")

print("\nYearly coverage:")
yearly_counts = df.groupby('year').size()
for year in [2023, 2024, 2025]:
    count = yearly_counts.get(year, 0)
    print(f"  {year}: {count:,} records")

print("\nMonthly coverage by year:")
monthly_counts = df.groupby([df['timestamp'].dt.year, df['timestamp'].dt.month]).size()
for year in [2023, 2024, 2025]:
    print(f"\n{year}:")
    for month in range(1, 13):
        if year == 2023 and month < 6:
            continue
        if year == 2025 and month > 6:
            break
        count = monthly_counts.get((year, month), 0)
        month_name = pd.Timestamp(f'{year}-{month:02d}-01').strftime('%B')
        print(f"  {month_name:10s}: {count:8,} records")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

daily_counts = df.groupby('date').size().reindex(all_dates, fill_value=0)
axes[0, 0].plot(daily_counts.index, daily_counts.values, linewidth=0.5, color='steelblue')
axes[0, 0].set_title('Daily Record Count (Jun 2023 - Jun 2025)', fontsize=12)
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Number of Records')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

for year in [2024, 2025]:
    year_start = pd.Timestamp(f'{year}-01-01')
    axes[0, 0].axvline(x=year_start, color='red', linestyle='--', alpha=0.5)

monthly_counts_plot = df.groupby(df['timestamp'].dt.to_period('M')).size()
axes[0, 1].bar(range(len(monthly_counts_plot)), monthly_counts_plot.values,
               color=['orange' if '2023' in str(idx) else 'steelblue' if '2024' in str(idx) else 'green'
                      for idx in monthly_counts_plot.index],
               edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Monthly Record Count', fontsize=12)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Number of Records')
axes[0, 1].set_xticks(range(0, len(monthly_counts_plot), 3))
axes[0, 1].set_xticklabels([str(idx) for idx in monthly_counts_plot.index][::3], rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

hourly_counts = df.groupby(df['timestamp'].dt.hour).size()
axes[1, 0].bar(hourly_counts.index, hourly_counts.values, color='steelblue', edgecolor='black')
axes[1, 0].set_title('Hourly Distribution of Records', fontsize=12)
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Number of Records')
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].grid(True, alpha=0.3, axis='y')

weekday_counts = df.groupby(df['timestamp'].dt.dayofweek).size()
weekday_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 1].bar(range(7), weekday_counts.reindex(range(7), fill_value=0).values,
               color='steelblue', edgecolor='black')
axes[1, 1].set_title('Records by Day of Week', fontsize=12)
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Number of Records')
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(weekday_names)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Temporal Coverage Analysis - OpenMeteo (Jun 2023 - Jun 2025)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Missing Data Analysis

In [ ]:
numerical_features = [
    'cloud_cover_pct_mean', 'dew_point_c_mean', 'humidity_pct_mean',
    'precipitation_mm_mean', 'pressure_hpa_mean', 'solar_radiation_wm2_mean',
    'temperature_c_mean'
]

available_features = [f for f in numerical_features if f in df.columns]

missing_stats = pd.DataFrame({
    'Missing Count': df[available_features].isnull().sum(),
    'Missing %': (df[available_features].isnull().sum() / len(df) * 100).round(2),
    'Available Count': df[available_features].notnull().sum(),
    'Available %': (df[available_features].notnull().sum() / len(df) * 100).round(2)
})

print("Missing Data Summary:")
print("="*60)
display(missing_stats)

## 4. Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

features_to_plot = [
    ('temperature_c_mean', 'Temperature (°C)', (-30, 45)),
    ('humidity_pct_mean', 'Humidity (%)', (0, 100)),
    ('precipitation_mm_mean', 'Precipitation (mm)', (0, 10)),
    ('pressure_hpa_mean', 'Pressure (hPa)', (980, 1040)),
    ('cloud_cover_pct_mean', 'Cloud Cover (%)', (0, 100)),
    ('dew_point_c_mean', 'Dew Point (°C)', (-30, 30)),
    ('solar_radiation_wm2_mean', 'Solar Radiation (W/m²)', (0, 1000)),
]

for idx, (feature, label, xlim) in enumerate(features_to_plot):
    if feature in df.columns:
        data = df[feature].dropna()
        data = data[np.isfinite(data)]
        
        if len(data) > 0:
            axes[idx].hist(data, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
            axes[idx].set_xlabel(label)
            axes[idx].set_ylabel('Frequency')
            axes[idx].set_title(f'{label} Distribution')
            if xlim:
                axes[idx].set_xlim(xlim)
            
            mean_val = data.mean()
            median_val = data.median()
            axes[idx].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.1f}')
            axes[idx].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.1f}')
            axes[idx].legend(loc='best', fontsize=9)
            axes[idx].grid(True, alpha=0.3)

for idx in range(len(features_to_plot), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Weather Feature Distributions (Jun 2023 - Jun 2025)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Geographic Coverage

In [ ]:
hex_locations = df[['h3_index_res8', 'h3_lat_res8', 'h3_lon_res8']].drop_duplicates()
print(f"Geographic Coverage:")
print("="*60)
print(f"Unique hexagons: {len(hex_locations):,}")
print(f"Latitude range: {hex_locations['h3_lat_res8'].min():.2f} to {hex_locations['h3_lat_res8'].max():.2f}")
print(f"Longitude range: {hex_locations['h3_lon_res8'].min():.2f} to {hex_locations['h3_lon_res8'].max():.2f}")

hex_data_counts = df.groupby('h3_index_res8').agg({
    'timestamp': ['count', 'min', 'max']
}).reset_index()
hex_data_counts.columns = ['h3_index_res8', 'record_count', 'first_date', 'last_date']
hex_with_counts = hex_locations.merge(hex_data_counts, on='h3_index_res8')

print(f"\nRecords per hexagon:")
print(f"  Mean: {hex_with_counts['record_count'].mean():.0f}")
print(f"  Median: {hex_with_counts['record_count'].median():.0f}")
print(f"  Min: {hex_with_counts['record_count'].min()}")
print(f"  Max: {hex_with_counts['record_count'].max()}")

expected_records = len(all_dates) * 24
completeness = hex_with_counts['record_count'] / expected_records * 100
print(f"\nData completeness per hexagon:")
print(f"  Mean: {completeness.mean():.1f}%")
print(f"  Median: {completeness.median():.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter = axes[0].scatter(hex_with_counts['h3_lon_res8'],
                         hex_with_counts['h3_lat_res8'],
                         c=hex_with_counts['record_count'],
                         cmap='YlOrRd', s=20, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('Weather Station Hexagon Locations with Data Density')
plt.colorbar(scatter, ax=axes[0], label='Record Count')

axes[1].hist(hex_with_counts['record_count'], bins=50, edgecolor='black', color='steelblue')
axes[1].set_xlabel('Records per Hexagon')
axes[1].set_ylabel('Number of Hexagons')
axes[1].set_title('Distribution of Records per Hexagon')
axes[1].axvline(expected_records, color='red', linestyle='--',
                label=f'Expected (full period): {expected_records}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Seasonal and Temporal Patterns

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['season'] = df['month'].map({12: 'Winter', 1: 'Winter', 2: 'Winter',
                                 3: 'Spring', 4: 'Spring', 5: 'Spring',
                                 6: 'Summer', 7: 'Summer', 8: 'Summer',
                                 9: 'Autumn', 10: 'Autumn', 11: 'Autumn'})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

monthly_temp = df.groupby('month')['temperature_c_mean'].mean()
axes[0, 0].plot(monthly_temp.index, monthly_temp.values, marker='o', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].set_title('Average Temperature by Month')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[0, 0].grid(True, alpha=0.3)

monthly_precip = df.groupby('month')['precipitation_mm_mean'].mean()
axes[0, 1].bar(monthly_precip.index, monthly_precip.values, color='steelblue', edgecolor='black')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Precipitation (mm)')
axes[0, 1].set_title('Average Precipitation by Month')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

monthly_humidity = df.groupby('month')['humidity_pct_mean'].mean()
axes[0, 2].plot(monthly_humidity.index, monthly_humidity.values, marker='s', linewidth=2, color='green', markersize=6)
axes[0, 2].set_xlabel('Month')
axes[0, 2].set_ylabel('Humidity (%)')
axes[0, 2].set_title('Average Humidity by Month')
axes[0, 2].set_xticks(range(1, 13))
axes[0, 2].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], rotation=45)
axes[0, 2].grid(True, alpha=0.3)

if 'season' in df.columns and df['season'].notna().any():
    seasonal_stats = df.groupby('season')[['temperature_c_mean', 'humidity_pct_mean',
                                           'precipitation_mm_mean']].mean()
    seasonal_stats.plot(kind='bar', ax=axes[1, 0])
    axes[1, 0].set_title('Seasonal Weather Patterns', fontsize=12)
    axes[1, 0].set_xlabel('Season')
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].legend(loc='best')
    axes[1, 0].tick_params(axis='x', rotation=0)

hourly_temp = df.groupby('hour')['temperature_c_mean'].mean()
axes[1, 1].plot(hourly_temp.index, hourly_temp.values, marker='o', linewidth=2)
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Temperature (°C)')
axes[1, 1].set_title('Average Temperature by Hour of Day')
axes[1, 1].set_xticks(range(0, 24, 2))
axes[1, 1].grid(True, alpha=0.3)

yearly_temp = df.groupby('year')['temperature_c_mean'].mean()
axes[1, 2].bar(yearly_temp.index, yearly_temp.values, color=['orange', 'steelblue', 'green'], edgecolor='black')
axes[1, 2].set_xlabel('Year')
axes[1, 2].set_ylabel('Temperature (°C)')
axes[1, 2].set_title('Average Temperature by Year')
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Seasonal and Temporal Weather Patterns (Jun 2023 - Jun 2025)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Yearly Comparison

In [ ]:
yearly_stats = df.groupby('year')[available_features].agg(['mean', 'std', 'min', 'max'])

print("Yearly Weather Statistics Comparison:")
print("="*60)
for feature in available_features:
    print(f"\n{feature.replace('_', ' ').title()}:")
    display(yearly_stats[feature].round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for year in [2023, 2024, 2025]:
    year_data = df[df['year'] == year]['temperature_c_mean'].dropna()
    if len(year_data) > 0:
        axes[0, 0].hist(year_data, bins=30, alpha=0.5, label=str(year), edgecolor='black')

axes[0, 0].set_xlabel('Temperature (°C)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Temperature Distribution by Year')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

yearly_monthly_temp = df.groupby(['year', 'month'])['temperature_c_mean'].mean().unstack(level=0)
yearly_monthly_temp.plot(ax=axes[0, 1], marker='o', linewidth=2)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Temperature (°C)')
axes[0, 1].set_title('Monthly Temperature Patterns by Year')
axes[0, 1].set_xticks(range(1, 13))
axes[0, 1].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[0, 1].legend(title='Year')
axes[0, 1].grid(True, alpha=0.3)

yearly_monthly_precip = df.groupby(['year', 'month'])['precipitation_mm_mean'].mean().unstack(level=0)
yearly_monthly_precip.plot(ax=axes[1, 0], marker='s', linewidth=2)
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Precipitation (mm)')
axes[1, 0].set_title('Monthly Precipitation Patterns by Year')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[1, 0].legend(title='Year')
axes[1, 0].grid(True, alpha=0.3)

yearly_monthly_humidity = df.groupby(['year', 'month'])['humidity_pct_mean'].mean().unstack(level=0)
yearly_monthly_humidity.plot(ax=axes[1, 1], marker='^', linewidth=2)
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Humidity (%)')
axes[1, 1].set_title('Monthly Humidity Patterns by Year')
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
axes[1, 1].legend(title='Year')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Yearly Weather Comparison (Jun 2023 - Jun 2025)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Feature Correlations

In [ ]:
correlation_matrix = df[available_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Weather Feature Correlation Matrix (Jun 2023 - Jun 2025)', fontsize=14)
plt.tight_layout()
plt.show()

print("\nStrongest correlations with temperature:")
if 'temperature_c_mean' in correlation_matrix.columns:
    temp_corr = correlation_matrix['temperature_c_mean'].sort_values(ascending=False)
    for feature, corr in temp_corr.items():
        if feature != 'temperature_c_mean' and abs(corr) > 0.3:
            print(f"  {feature}: {corr:.3f}")

## 9. Statistical Summary

In [ ]:
print("Statistical Summary of Weather Features (Full Dataset):")
print("="*60)
display(df[available_features].describe())

In [ ]:
print("OPENMETEO EXTENDED DATASET SUMMARY (Jun 2023 - Jun 2025)")
print("="*60)

print("\n📊 DATASET OVERVIEW:")
print(f"   Total records: {len(df):,}")
print(f"   Time period: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")
print(f"   Unique locations (hexagons): {df['h3_index_res8'].nunique()}")
print(f"   Temporal resolution: Hourly")

print("\n📅 TEMPORAL COVERAGE:")
print(f"   Days with data: {len(existing_dates)}/{len(all_dates)} ({len(existing_dates)/len(all_dates)*100:.1f}%)")
print(f"   Data period: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
if len(existing_dates) == len(all_dates):
    print("   ✓ COMPLETE COVERAGE FOR ANALYSIS PERIOD")

print("\n🌡️ KEY WEATHER STATISTICS:")
if 'temperature_c_mean' in df.columns:
    print(f"   Temperature range: {df['temperature_c_mean'].min():.1f}°C to {df['temperature_c_mean'].max():.1f}°C")
    print(f"   Mean temperature: {df['temperature_c_mean'].mean():.1f}°C")
if 'humidity_pct_mean' in df.columns:
    print(f"   Mean humidity: {df['humidity_pct_mean'].mean():.1f}%")
if 'precipitation_mm_mean' in df.columns:
    print(f"   Mean precipitation: {df['precipitation_mm_mean'].mean():.2f} mm")
if 'pressure_hpa_mean' in df.columns:
    print(f"   Mean pressure: {df['pressure_hpa_mean'].mean():.1f} hPa")

print("\n✅ DATA COMPLETENESS:")
for feature in available_features:
    completeness = df[feature].notna().mean() * 100
    print(f"   {feature}: {completeness:.1f}%")

print("\n💾 DATASET READY FOR:")
print("   • PM2.5 enrichment with weather features")
print("   • Multi-year weather analysis")
print("   • Spatial-temporal modeling")
print("   • Air quality prediction models")
print("   • Climate trend analysis")